In [ ]:
using LinearAlgebra
using Random
using Statistics
using Printf
using Plots

### Problem 1.


In [6]:
# a) Implement the clgs algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code is given in Algorithm 7.1. 
# The function should output matrices Q and R.
# Solution: Note that here, the algorithm computes the reduced QR since, we can always get to Full QR by adding more
# orthogonal columns to Q and simply adding a zero block to R.

function clgs(A::AbstractMatrix{<:Real})
    m, n = size(A)
    Q̂ = zeros(Float64, m, n)
    R̂ = zeros(Float64, n, n)
    for j in 1:n
        v = Float64.(A[:, j])
        for i in 1:(j-1)
            R̂[i, j] = dot(Q̂[:, i], A[:, j])
            v -= R̂[i, j] * Q̂[:, i]
        end
        R̂[j, j] = norm(v)
        Q̂[:, j] = v / R̂[j, j]
    end
    return Q̂, R̂
end

clgs (generic function with 1 method)

In [ ]:
#Test Block for algorithm correctness:

Random.seed!(3)
A = Float64.(rand(-5:5, 4, 3))
display(A)
Q, R = clgs(A)
Ā = Q * R
ϵ = Ā - A

display("Q:")
display(Q)
display("R:")
display(R)
display("Ā = QR:")
display(Ā)
display("ϵ = Ā - A")
display(ϵ) #Some Floating Point Error is Noticed where the matrix entry was originally zero, rest matches original.

4×3 Matrix{Float64}:
 -2.0  2.0   3.0
  5.0  0.0   0.0
  5.0  5.0  -2.0
 -4.0  3.0   5.0

"Q:"

4×3 Matrix{Float64}:
 -0.239046   0.371863   0.282791
  0.597614  -0.10591    0.793219
  0.597614   0.717836  -0.356111
 -0.478091   0.578976   0.404991

"R:"

3×3 Matrix{Float64}:
 8.3666  1.07571  -4.30282
 0.0     6.06983   2.57479
 0.0     0.0       3.58555

"Ā. = QR:"

4×3 Matrix{Float64}:
 -2.0  2.0           3.0
  5.0  4.09145e-17  -1.13425e-16
  5.0  5.0          -2.0
 -4.0  3.0           5.0

"ϵ = Ā - A"

4×3 Matrix{Float64}:
 0.0  0.0           0.0
 0.0  4.09145e-17  -1.13425e-16
 0.0  0.0           0.0
 0.0  0.0           0.0

In [ ]:
#b)Compute the clgs of the 2-by-2 matrix A as in equation (9.1) of your book (page 67).Test orthogonality by computing the matrix norm of the matrix Q^T Q − I_n,
# where I_n is n-by-n identity matrix. Compute both the 2-norm and the Frobenius norm of this matrix, and explain the result.
# Solution: using the fucntion clgs computed in a) we have:

A = Float64.([0.70000    0.70711;
     0.70001    0.70711])
display(A)
Q, R = clgs(A)
Ā = Q * R

display("Q")
display(Q)
display("R")
display(R)
display("Reconstruction → Ā = QR")
display(Ā)

#Testing Orthogonality we have:
Iₙ = I(2)
ϵ = Q'Q - Iₙ
display("ϵ =Q'Q - Iₙ ")
display(ϵ)

F = norm(ϵ)
L = opnorm(ϵ)

display("F-Norm:")
display(F)

display("L-2 Norm:")
display(L)

q₁ = Q[:,1]
q₂ = Q[:,2]

display(q₁)
display(q₂)

cosθ = (q₁ ⋅ q₂) / (norm(q₁) * norm(q₂))
θ = acosd(clamp(cosθ, -1, 1))
display(θ)
display(90 - θ)


2×2 Matrix{Float64}:
 0.7      0.70711
 0.70001  0.70711

"Q"

2×2 Matrix{Float64}:
 0.707102   0.707112
 0.707112  -0.707102

"R"

2×2 Matrix{Float64}:
 0.989957  1.0
 0.0       7.14284e-6

...

2.3014368188967183e-11

2-element Vector{Float64}:
 0.7071017304418633
 0.7071118318951554

2-element Vector{Float64}:
  0.7071118319114289
 -0.7071017304255895

89.99999999868137

1.318625209023594e-9

#### (b) Explanation:

Here see that $||\epsilon||_2 = 2.3014368188967183e-11$ and $||\epsilon||_F = 3.2547231622285317e-11$. Both norms are non-zero up to degree $10^{-11}$, indicating a non-zero error matrix $\epsilon = Q'Q -I$. However, as per the properties of orthonormal matrices, we know $Q^*Q = I \quad \Rightarrow Q^*Q - I = 0 \quad \Rightarrow ||Q^*Q - I|| = ||0|| = 0$.\\

See that the diagonal entries of $\epsilon$ are zero with non-zero entries on the off-diagonal entries, indicating $q_1, q_2 \text{ of } Q$ computed through the QR decomp are not perfectly orthogonal, the angle between them can be computed as:
$$\cos^{-1}(\frac{q_1^* q_2}{||q_1||^* ||q_2||}) = 89.99999999868137$$
showcasing an error of $1.318625209023594e-9$. The columns of $Q$ have unit length to machine precision, but they are **not exactly orthogonal**.

During each step of the clgs computation, floating point arithmetic rounds every operation to about $10^{-16}$ relative accuracy, which compounds as the algorithm processes the next column. However, our orthogonality error is about $10^{-11}$ showing an amplification of about $10^{5}$. This is likely because entries of A are perturbations of around the same number $0.7$, which cancels out columns, introducing floating point numbers during the computation process.

### Problem 2

In [ ]:
# Modified Gram Schmidt (MGS) iteration
# Implement the mgs algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code
# is given in Algorithm 8.1. The function should output matrices Q and R.
function mgs(A::AbstractMatrix{<:Real})
    m, n = size(A)
    V = Float64.(copy(A))
    Q = zeros(Float64, m,n)
    R = zeros(Float64, n,n)
    for i in 1:n
        R[i,i] = norm(V[:, i])
        Q[:, i] = V[:,i]/ R[i,i]
        for j in (i+1):n
            R[i,j] = dot(Q[:, i], V[:, j])
            V[:, j] -= R[i,j]* Q[:,i]
        end
    end
    return Q, R
end


mgs (generic function with 1 method)

In [ ]:
# HouseHolder Triangularization
# b) Implement the house algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code
# is given in Algorithm 10.1 (follow the book’s advice about using Algorithm 10.3 to
# construct Q). The function should output matrices Q and R.

function house(A::AbstractMatrix{<:Real})
    m,n = size(A)
    R = Float64.(copy(A))
    W = zeros(Float64, m, n)
    for k in 1:n
        x = R[k:m, k]
        v = copy(x)
        v[1] += (x[1] >= 0 ? 1 : -1) * norm(x)
        v /= norm(v, 2)
        R[k:m, k:n] -= 2 * v * (v' * R[k:m, k:n])
        W[k:m,k] = v
    end
    Q = formQ(W)
    return Q, triu(R)
end

function formQ(W::AbstractMatrix{<:Real})
    m, n = size(W)
    Q = Matrix{Float64}(I, m, m)
    for j in 1:m
        x = Q[:, j]
        for k in n:-1:1
            v = W[k:m, k]
            x[k:m] -= 2 * v * (v' * x[k:m])
        end
        Q[:, j] = x
    end
    return Q
end

formQ (generic function with 1 method)

In [ ]:
# c) Implement the givens algorithm you derived in exercise 10.4. The function should
# output matrices Q and R
# Solution: for each column j, zero out the entries below the diagonal from the bottom up, using a rotation
# G = [c s; -s c] on rows (i-1, i) with c = a/r, s = b/r, r = √(a² + b²), which maps (a, b) → (r, 0).

function givens(A::AbstractMatrix{<:Real})
    m, n = size(A)
    R = Float64.(copy(A))
    Q = Matrix{Float64}(I, m, m)
    for j in 1:n
        for i in m:-1:(j+1)
            a, b = R[i-1, j], R[i, j]
            b == 0 && continue
            r = hypot(a, b)
            c, s = a / r, b / r
            G = [c s; -s c]
            R[[i-1, i], j:n] = G * R[[i-1, i], j:n]
            Q[:, [i-1, i]] = Q[:, [i-1, i]] * G'
        end
    end
    return Q, triu(R)
end

In [ ]:
function make_Q0(m::Int, n::Int)
    F = qr(randn(m, m))
    Qfull = Matrix(F.Q) * Diagonal(sign.(diag(F.R)))
    return Qfull[:, 1:n]
end

function make_R0(n::Int; σ = 0.1)
    S0 = Diagonal([2.0^(-j) for j in 1:n])
    R = triu(σ * randn(n, n), 1) + I
    return S0 * R
end

function make_A(m::Int, n::Int)
    Q0 = make_Q0(m, n)
    R0 = make_R0(n)
    A = Q0 * R0
    return A, Q0, R0
end

In [ ]:
# d) Construct A = Q₀R₀ of size m-by-m, m = n = 40, with Q₀ a random orthogonal matrix and R₀ = S₀R, where
# S₀ = diag(2⁻ʲ) and R is upper triangular with Rⱼⱼ = 1, Rᵢⱼ ~ N(0, 0.1²) i.i.d. for j > i.
# Solution: generate the true factors first, then form A = Q₀R₀. Q0 and R0 store the ground truth.

Random.seed!(561)
m = n = 40
A, Q0, R0 = make_A(m, n)

display("True Q₀ (random orthogonal matrix):")
display(Q0)
display("True R₀ = S₀R (upper triangular, diagonal 2⁻ʲ):")
display(R0)
display("A = Q₀R₀:")
display(A)
display("cond(A):")
display(cond(A))

In [ ]:
# Helpers (also used in parts f, g):
#  - lapack_qr: Julia's default qr (LAPACK), returned as ordinary matrices Q, R
#  - reduce_and_fix: converts the full QR (house, givens) to the reduced QR and flips signs so that
#    diag(R) > 0, since the QR factorization is only unique with a positive diagonal. This lets us
#    compare every method's output directly with the true Q₀, R₀.

function lapack_qr(A::AbstractMatrix{<:Real})
    F = qr(A)
    return Matrix(F.Q), Matrix(F.R)
end

function reduce_and_fix(Q::AbstractMatrix, R::AbstractMatrix)
    n = size(R, 2)
    Q̂, R̂ = Q[:, 1:n], R[1:n, :]
    D = Diagonal(sign.(diag(R̂)))
    return Q̂ * D, D * R̂
end

methods_list = [("clgs", clgs), ("mgs", mgs), ("house", house), ("givens", givens), ("LAPACK qr", lapack_qr)]

In [ ]:
# e) Evaluate the error of the orthogonality of the Q output by the different algorithms, by computing the
# norm of QᵀQ − I. Specify the norm being used.
# Solution: we compute ϵ = QᵀQ − I for each method and measure it with two norms:
#  - the 2-norm ‖ϵ‖₂ (opnorm): the largest amount by which Q can stretch or shrink a vector
#  - the Frobenius norm ‖ϵ‖_F (norm): the total size of all the entries of ϵ
# Uses Q_clgs, Q_mgs, Q_house, Q_givens, Q_lapack computed in the cells above.

# CLGS
ϵ = Q_clgs'Q_clgs - I
display("CLGS → 2-norm ‖QᵀQ − I‖₂:")
display(opnorm(ϵ))
display("CLGS → Frobenius norm ‖QᵀQ − I‖_F:")
display(norm(ϵ))

# MGS
ϵ = Q_mgs'Q_mgs - I
display("MGS → 2-norm ‖QᵀQ − I‖₂:")
display(opnorm(ϵ))
display("MGS → Frobenius norm ‖QᵀQ − I‖_F:")
display(norm(ϵ))

# HOUSE
ϵ = Q_house'Q_house - I
display("HOUSE → 2-norm ‖QᵀQ − I‖₂:")
display(opnorm(ϵ))
display("HOUSE → Frobenius norm ‖QᵀQ − I‖_F:")
display(norm(ϵ))

# GIVENS
ϵ = Q_givens'Q_givens - I
display("GIVENS → 2-norm ‖QᵀQ − I‖₂:")
display(opnorm(ϵ))
display("GIVENS → Frobenius norm ‖QᵀQ − I‖_F:")
display(norm(ϵ))

# LAPACK qr
ϵ = Q_lapack'Q_lapack - I
display("LAPACK qr → 2-norm ‖QᵀQ − I‖₂:")
display(opnorm(ϵ))
display("LAPACK qr → Frobenius norm ‖QᵀQ − I‖_F:")
display(norm(ϵ))

#### (e) Explanation:

Here we measure $\epsilon = Q^TQ - I$ with the 2-norm $||\epsilon||_2$ (the largest amount by which $Q$ can stretch or shrink a vector) and the Frobenius norm $||\epsilon||_F$ (the total size of all entries of $\epsilon$). If $Q$ were perfectly orthogonal, both would be $0$. Our matrix has cond(A) $\approx 7.4 \times 10^{11}$, i.e. its columns are very close to being linearly dependent.

**CLGS: Explanation**

See that $||\epsilon||_2 = 8.54$ and $||\epsilon||_F = 9.04$, both bigger than 1, indicating that $Q$ has completely lost orthogonality. As in Problem 1(b), each step subtracts nearly parallel vectors and divides by a tiny $r_{jj}$, which blows up the rounding error, and since clgs projects against the original column $a_j$, these errors are never removed and keep compounding.

**MGS: Explanation**

Here $||\epsilon||_2 = 8.93 \times 10^{-6}$ and $||\epsilon||_F = 1.32 \times 10^{-5}$, much better than clgs since mgs removes earlier rounding errors by projecting against the updated vector, but still far above machine precision ($\approx 10^{-16}$), because the loss of orthogonality still grows with the condition number of $A$.

**House, Givens and LAPACK: Explanation**

All three give $||\epsilon||_2 \approx 2 \times 10^{-15}$ and $||\epsilon||_F \approx 5$–$6 \times 10^{-15}$, i.e. orthogonal up to machine precision. This is because $Q$ is built as a product of reflections or rotations, each exactly orthogonal, rather than by subtracting projections, so it stays orthogonal regardless of how ill-conditioned $A$ is.

**2-norm vs. Frobenius norm**

In every case $||\epsilon||_2 \leq ||\epsilon||_F$, as expected. For clgs the two are almost equal, showing that its error is concentrated in essentially one direction (many columns collapsing onto the same direction), while for house, givens and LAPACK the Frobenius norm is about $2.5$ times larger, showing that their tiny rounding errors are spread out across many directions.

In [ ]:
# f) Evaluate the errors of Q and R compared to the ground truth {Q₀, R₀} for the different algorithms.
# Compute 20 random realizations of A and report the mean and standard deviation of the error.
# Errors used (all 2-norms): ‖Q - Q₀‖₂ (‖Q₀‖₂ = 1, so absolute = relative), ‖R - R₀‖₂/‖R₀‖₂,
# the backward error ‖A - QR‖₂/‖A‖₂, and the orthogonality error ‖QᵀQ - I‖₂.

function qr_errors(A, Q0, R0, f)
    Q, R = reduce_and_fix(f(A)...)
    return (opnorm(Q - Q0), opnorm(R - R0) / opnorm(R0), opnorm(A - Q * R) / opnorm(A), opnorm(Q'Q - I))
end

function run_trials(m, n, trials)
    errs = Dict(name => zeros(trials, 4) for (name, _) in methods_list)
    for t in 1:trials
        A, Q0, R0 = make_A(m, n)
        for (name, f) in methods_list
            errs[name][t, :] .= qr_errors(A, Q0, R0, f)
        end
    end
    return errs
end

function print_table(errs)
    @printf("%-10s  %-21s  %-21s  %-21s  %-21s\n", "method", "‖Q−Q₀‖₂", "‖R−R₀‖₂/‖R₀‖₂", "‖A−QR‖₂/‖A‖₂", "‖QᵀQ−I‖₂")
    for (name, _) in methods_list
        E = errs[name]
        @printf("%-10s", name)
        for c in 1:4
            @printf("  %.2e ± %.2e", mean(E[:, c]), std(E[:, c]))
        end
        println()
    end
end

Random.seed!(561)
errs = run_trials(40, 40, 20)
println("m = n = 40  (mean ± std over 20 realizations)")
print_table(errs)

#### (f) Explanation:

Here we compare the computed $Q$ and $R$ against the true factors $Q_0, R_0$ over 20 random matrices with $m = n = 40$, looking at four errors (all in the 2-norm): $||Q - Q_0||_2$, $||R - R_0||_2 / ||R_0||_2$, the reconstruction error $||A - QR||_2 / ||A||_2$, and the orthogonality error $||Q^TQ - I||_2$. The values quoted below are the means, the standard deviations are printed in the output above.

**Reconstruction of A (all methods)**

See that every method, including clgs, reconstructs $A$ almost perfectly, with $||A - QR||_2 / ||A||_2$ between $6 \times 10^{-17}$ and $7 \times 10^{-16}$. So simply checking that $QR$ gives back $A$ is not enough to tell whether an algorithm works well, since clgs passes this test even though its $Q$ is completely wrong.

**CLGS: Explanation**

Here $||Q - Q_0||_2 \approx 3.0$ and $||Q^TQ - I||_2 \approx 8.7$, indicating that $Q$ is neither orthogonal nor anywhere close to the true $Q_0$. $R$ is also off, with a relative error of about $2.9 \times 10^{-8}$, which is much larger than the other methods (around $10^{-16}$). The errors in $Q$ and $R$ cancel each other out when multiplied, which is why the product $QR$ still matches $A$.

**MGS: Explanation**

Here $||Q - Q_0||_2 \approx 2.0 \times 10^{-5}$, which is about the same size as its orthogonality error ($2.0 \times 10^{-5}$), showing that most of the error in $Q$ comes from $Q$ not being orthogonal. $R$ on the other hand is accurate, with a relative error of $2.4 \times 10^{-16}$.

**House, Givens and LAPACK: Explanation**

These give a $Q$ that is orthogonal to about $2 \times 10^{-15}$ and an $R$ accurate to about $3 \times 10^{-16}$, but surprisingly $||Q - Q_0||_2$ is still around $10^{-6}$ (house $3.1 \times 10^{-6}$, givens $1.8 \times 10^{-6}$, LAPACK $4.8 \times 10^{-6}$), much bigger than machine precision. This is not a fault of the algorithms. Since cond(A) $\approx 10^{12}$, the $Q$ factor itself is very sensitive to small changes in $A$, so even the tiny rounding errors of a stable algorithm get amplified into a visible change in $Q$. The errors in $Q$ and $R$ are tied together in a way that cancels in the product, which is why $QR$ still matches $A$ to machine precision. In other words, a good algorithm guarantees that $QR$ is accurate, but not that $Q$ and $R$ individually match the true factors.

Note that the relative error in $R$ is dominated by its first few rows, which are much larger than the last rows (scaled by $2^{-1}$ vs. $2^{-40}$), so a small value here does not mean that every entry of $R$ is accurate.

**Summary**

Overall, clgs fails completely, mgs gives a $Q$ that is noticeably not orthogonal, and house, givens and LAPACK give an orthogonal $Q$ up to machine precision. For ill-conditioned matrices like ours, house, givens or LAPACK should be used, mgs only when $A$ is reasonably well conditioned, and clgs should be avoided.

In [ ]:
# g) Let A be of size m-by-n, n = 40, m = 80, 160, 320, 640. Q₀ consists of the first n columns of a random
# m-by-m orthogonal matrix, while R₀ is constructed as before. How does this affect the results in (d)?
# Solution: make_A already takes the first n columns of a random m×m orthogonal matrix, and reduce_and_fix
# converts the full QR (house, givens) to the reduced QR, which is the analogue of qr(A, "econ").

ms = [80, 160, 320, 640]
Random.seed!(561)
errs_g = Dict(m => run_trials(m, 40, 20) for m in ms)

for m in ms
    println("\nm = $m, n = 40  (mean ± std over 20 realizations)")
    print_table(errs_g[m])
end

method_names = first.(methods_list)
panels = [(1, "‖Q − Q₀‖₂"), (4, "‖QᵀQ − I‖₂"), (3, "‖A − QR‖₂ / ‖A‖₂")]
plts = map(panels) do (c, t)
    p = plot(xscale = :log2, yscale = :log10, xticks = (ms, string.(ms)),
             xlabel = "m  (n = 40)", ylabel = "mean error", title = t, legend = :right)
    for name in method_names
        plot!(p, ms, [mean(errs_g[m][name][:, c]) for m in ms], marker = :o, label = name)
    end
    p
end
plot(plts..., layout = (1, 3), size = (1300, 400), margin = 5Plots.mm)

#### (g) Explanation:

Here we repeat the experiment from (f) for tall matrices with $n = 40$ and $m = 80, 160, 320, 640$ (see the printed output and plot above). Since $Q_0$ has orthonormal columns, cond(A) = cond($R_0$) $\approx 10^{12}$ for every $m$, so making $A$ taller does not make it any better or worse conditioned. As a result, the overall picture from (d)–(f) stays the same.

**CLGS: Explanation**

Nothing changes, for every $m$ we still get $||Q^TQ - I||_2 \approx 9$ and $||Q - Q_0||_2 \approx 3$, so $Q$ is still completely wrong.

**MGS: Explanation**

The orthogonality error grows slightly with $m$, from $2.5 \times 10^{-5}$ at $m = 80$ to $4.5 \times 10^{-5}$ at $m = 640$. Longer columns mean more arithmetic in every projection and hence a bit more rounding error, but the main cause of the error is still the ill-conditioning of $A$.

**House, Givens and LAPACK: Explanation**

$Q$ stays orthogonal up to machine precision (about $2$–$4 \times 10^{-15}$), and the reconstruction error stays around $10^{-15}$, both creeping up slowly with $m$ for the same reason as above. Of the three, LAPACK has the smallest orthogonality and reconstruction errors at every $m$.

The one noticeable change is in $||Q - Q_0||_2$, which jumps about ten times from the square case ($\approx 3 \times 10^{-6}$) to $m = 80$ ($\approx 3 \times 10^{-5}$), and then grows slowly. When $A$ is square, $Q$ is an $m \times m$ orthogonal matrix, so its columns already span the whole space and rounding errors can only rotate the columns among themselves. When $m > n$, the columns of $Q$ only span an $n$-dimensional subspace of $\mathbb{R}^m$, so rounding errors can now also tilt this subspace into the extra $m - n$ directions, and these errors get amplified by the ill-conditioning of $A$. This makes $Q$ more sensitive for tall matrices.

**Summary**

Making $A$ taller does not change the conclusions: clgs fails, mgs loses orthogonality because of the ill-conditioning of $A$, and house, givens and LAPACK stay orthogonal up to machine precision, with all errors increasing only slightly with $m$.